<!-- notebook-header -->
<a id="top"></a>

# 11. Design Problems

*LeetCode Playbook · notebook 11 of 12* · From requirements to a class: one structure per operation, kept in sync by an invariant.

**In this notebook:** [Design Problems: From Requirements to Classes](#s24)

← [Matrices, Math & Bits](10_Matrices_Math_Bits.ipynb) · [All notebooks](01_Start_Here.ipynb#notebooks) · [A-Z problem finder](12_Interview_Day.ipynb#finder) · [Interview Day](12_Interview_Day.ipynb) →

Run the setup cell below first; after that every section runs from its first cell.

In [ ]:
# Setup: everything this notebook imports. Run this cell first.
from collections import Counter, defaultdict, deque, OrderedDict
from functools import cmp_to_key, lru_cache
from itertools import accumulate, combinations, permutations, product
import bisect, heapq, math, random, string

<a id="s24"></a>

## Design Problems: From Requirements to Classes

> A design problem is a contract: a list of operations, each with a price ("O(1)", "O(log n)"). Give every operation a structure that answers its question at that price, write down the one sentence that keeps the structures in sync, and the methods nearly write themselves.

**Reach for it when** the prompt says *Design ...* or *Implement a class ...*, describes a **tracker, counter, rate limiter, cache or store**, lists method signatures, or puts a cost on every call ("each in O(1)", "many queries after one construction").

**In this repo:** `design/` (22 problems) · bank: `practice/simple/24_lru_cache.py` · basics: `practice/simple/basics/stacks/01_array_stack_and_queue_via_two_stacks.py` (a queue built from two stacks, the smallest design problem there is) · taught in other sections: Min Stack (`stack/min_stack.py`, `practice/simple/14_min_stack.py`, Stacks & Queues) and MedianFinder (`heap/find_median_from_data_stream.py`, `practice/simple/34_find_median_from_data_stream.py`, Heaps)

### The picture

```text
THE METHOD, drawn for LRU Cache

  operations and their price      one structure per question             the sentence that ties them
  get(key)         O(1)  ---->    "where is key?"    dict key -> node    the keys in the dict are exactly
  put(key, value)  O(1)  ---->    "who is oldest?"   doubly linked list  the nodes in the list, and the
  evict when full  O(1)  ---->    "make it newest"   in recency order    list order is the recency order

  map   { 1: *, 2: *, 3: * }      each * points straight at a node in the list
  list  HEAD <-> [3] <-> [1] <-> [2] <-> TAIL
                 newest           oldest: evicted next
  get(2):    unlink [2], re-link it right after HEAD    ->  HEAD <-> [2] <-> [3] <-> [1] <-> TAIL
  put(4, d): link [4] after HEAD; too full  ->  unlink TAIL.prev ([1]) and delete map[1]
```

Why it is fast: the brute force keeps one list of `(key, value)` pairs in recency order, so every `get` scans it (O(n)) and moving a pair to the front shifts everything behind it (O(n)). The dict removes the scan. The doubly linked list removes the shift: a node knows both of its neighbours, so unlinking it is two pointer writes, and the dict hands us the node directly. Every method becomes a constant number of dict operations and pointer moves.

### From idea to code

**The idea in one sentence:** *for each operation, ask which question it must answer and pick the structure that answers that question at the required price; then write the invariant that keeps all the structures describing the same items.*

The method, in the order you type:

1. **List the operations and their price.** Write them as a comment at the top of the class.
2. **Pick one structure per question** (the table below covers almost every design problem).
3. **Write the invariant** that keeps the structures in sync, as a comment in `__init__`.
4. **Write the private helpers first** (`_unlink`, `_push_front`, `_evict`, `_touch`): the small verbs every public method shares. Each helper leaves the invariant true.
5. **Then the public methods**, each a short sequence of helper calls.
6. **Walk the edge cases out loud**: capacity 0 or 1, a key that already exists (update *and* refresh), a missing key, an empty structure, duplicates, equal timestamps.

| The operation needs... | Structure | Cost |
|---|---|---|
| find an item by key | `dict` | O(1) |
| usage order: move any item to the front, drop the oldest | doubly linked list with sentinels, or `OrderedDict` | O(1) |
| a sliding time window: add new events, drop expired ones | `deque` + a running total | O(1) amortized |
| the smallest or largest, with inserts | heap (`heapq`) | O(log n) |
| delete arbitrary items from a heap | heap + lazy deletion (skip stale tops) | O(log n) amortized |
| a uniformly random element | dense `list` + `dict` value → index | O(1) |
| "the value as of time t" | per-key list of (time, value) appended in time order + `bisect` | O(log n) |
| a sorted set of disjoint intervals | sorted list(s) + `bisect` | O(log n) search, O(n) insert |
| prefix sums with point updates | Fenwick tree | O(log n) |
| paths and prefixes | trie of dicts | O(length) |
| min or max of counts that change by ±1 | count buckets + a pointer to the min or max count | O(1) |

The seven decisions, answered for a design problem (LRU Cache):

| Decision | Design answer (LRU Cache) |
|---|---|
| **State**: what must I remember? | the structures: `self.map` (a dict), a doubly linked list between the sentinels `self.head` and `self.tail`, and `self.cap` |
| **Definition**: what does each one hold? | `map[key]` = the node holding key's value; list order = recency (`head.next` newest, `tail.prev` oldest); every node stores its own key |
| **Invariant**: how do they stay in sync? | the keys in `map` are exactly the nodes in the list, and after every public call `len(map) <= cap` |
| **Step**: what does each operation do? | `get`: move the node to the front. `put` of an existing key: update the value and move it to the front. `put` of a new key: link a new node at the front, then evict `tail.prev` if over capacity |
| **Record**: what does each method return? | `get` returns the value or `-1`; `put` returns nothing |
| **Init**: the constructor | an empty dict; `head` and `tail` linked to each other, so even an empty list has no `None` ends |
| **Return / edge cases** | capacity 1; `put` of an existing key (refresh, no eviction); `get` of a missing key; capacity 0 (LeetCode promises ≥ 1, but know what your code does) |

The same design, sentence by sentence:

| In words | In code |
|---|---|
| "find the key's node" | `node = self.map.get(key)` |
| "take it out of the line" | `node.prev.next, node.next.prev = node.next, node.prev` |
| "put it at the front (newest)" | `node.prev, node.next = self.head, self.head.next`, then `self.head.next.prev = node` and `self.head.next = node` |
| "the oldest item" | `self.tail.prev` |
| "forget it completely" | unlink it and `del self.map[victim.key]` (this is why a node stores its key) |
| "too full" | `len(self.map) > self.cap` |
| "an empty list with no None checks" | `self.head.next, self.tail.prev = self.tail, self.head` |

Helpers first, then the public methods. The tags are the decisions above:

In [ ]:
class Node:
    __slots__ = ("key", "val", "prev", "next")

    def __init__(self, key=0, val=0):
        self.key, self.val = key, val            # the key lets eviction delete the map entry
        self.prev = self.next = None


class LRUCache:
    """get(key), put(key, value): both O(1). When full, evict the least recently used key."""

    def __init__(self, capacity):                # INIT
        self.cap = capacity
        self.map = {}                            # STATE: key -> Node
        self.head, self.tail = Node(), Node()    # STATE: sentinels; head.next = newest, tail.prev = oldest
        self.head.next, self.tail.prev = self.tail, self.head
        # INVARIANT: the keys in self.map are exactly the nodes between head and tail

    def _unlink(self, node):                     # helpers: a few pointer moves, no None checks
        node.prev.next, node.next.prev = node.next, node.prev

    def _push_front(self, node):
        node.prev, node.next = self.head, self.head.next
        self.head.next.prev = node
        self.head.next = node

    def _evict(self):
        victim = self.tail.prev                  # the least recently used node
        self._unlink(victim)
        del self.map[victim.key]                 # keep the dict in sync with the list

    def get(self, key):
        node = self.map.get(key)
        if node is None:
            return -1                            # RETURN: missing key
        self._unlink(node)                       # STEP: a read is a use -> move to the front
        self._push_front(node)
        return node.val                          # RECORD

    def put(self, key, value):
        node = self.map.get(key)
        if node is not None:                     # STEP: existing key -> update AND refresh
            node.val = value
            self._unlink(node)
        else:
            node = self.map[key] = Node(key, value)
        self._push_front(node)
        if len(self.map) > self.cap:             # insert first, then evict: capacity 0 also works
            self._evict()


cache = LRUCache(2)
cache.put(1, 1)
cache.put(2, 2)
print(cache.get(1))                              # 1
cache.put(3, 3)                                  # evicts 2: key 1 was used more recently
print(cache.get(2))                              # -1
cache.put(4, 4)                                  # evicts 1
print(cache.get(1), cache.get(3), cache.get(4))  # -1 3 4

**Try it**
- Delete the line `del self.map[victim.key]` and rerun: the second print shows `2` instead of `-1`. The dict still points at a node that left the list; the invariant broke, and the answer is wrong without any error.
- Move the two eviction lines above `self._push_front(node)` (evict first, as `practice/simple/24_lru_cache.py` does) and run `LRUCache(0).put(1, 1)`: `AttributeError`. In an empty list `tail.prev` is the head sentinel, whose `prev` is `None`. With capacity ≥ 1 both orders work.
- Predict, then check: `c = LRUCache(2)`, `put(1, 1)`, `put(2, 2)`, `put(1, 10)`, `put(3, 3)`. Key 2 is evicted, because updating key 1 also refreshed it: `c.get(2)` is -1 and `c.get(1)` is 10.

### Watch it work

The list from newest to oldest after each call:

In [ ]:
def recency(c):                                  # keys from newest to oldest
    keys, node = [], c.head.next
    while node is not c.tail:
        keys.append(node.key)
        node = node.next
    return keys


c = LRUCache(2)
for op, *args in [("put", 1, 1), ("put", 2, 2), ("get", 1), ("put", 3, 3),
                  ("get", 2), ("put", 1, 10), ("get", 3)]:
    result = getattr(c, op)(*args)
    call = f"{op}({', '.join(map(str, args))})"
    print(f"{call:10} -> {str(result):4}  newest..oldest {recency(c)}")

**Try it**
- Add `("put", 4, 4)` at the end of the list and predict which key leaves before running: 1, because `get(3)` made 3 the newest.
- Print `sorted(c.map)` next to `recency(c)`: always the same keys. That is the invariant, made visible.
- Run it with `LRUCache(1)`: the list never holds more than one key, and every `get` returns -1, because each one asks for a key that the previous `put` pushed out.

### The method again: LFU Cache

LFU (460) changes one requirement: evict the key with the **fewest uses**, and among those the least recently used one. Run the method again:

| Operation and price | The question it asks | Structure |
|---|---|---|
| `get`, `put` of an existing key: O(1) | where is the key, and how many uses does it have? | `freq_of = {key: count}` |
| one more use: O(1) | move the key from bucket f to bucket f + 1, as the newest there | `bucket = {count: OrderedDict(key -> value)}`, each bucket in LRU order |
| evict: O(1) | which count is the smallest, and who is oldest in it? | `self.min_freq`, then `bucket[min_freq].popitem(last=False)` |

**Invariant:** every key lives in exactly one bucket, `bucket[freq_of[key]]`, and `min_freq` is the smallest non-empty bucket whenever the cache is not empty.

Why one integer can track the minimum: counts only ever go **up by one**. So the minimum changes in just two ways: a brand-new key arrives (the minimum becomes 1), or the last key of the minimum bucket moves up (the minimum becomes `min_freq + 1`, exactly where that key went). An eviction can empty the lowest floor too, but it only ever happens right before a new key arrives, so the first case covers it.

```text
counts as floors, each floor a queue (oldest on the left):

freq 1: [2]        freq 2: [1, 3]        min_freq = 1
get(2):   2 climbs to floor 2; floor 1 is now empty and was the min   ->  min_freq = 2
freq 2: [1, 3, 2]                        min_freq = 2
put(4) when full: evict the oldest on the lowest floor (1), THEN 4 enters floor 1   ->  min_freq = 1
freq 1: [4]        freq 2: [3, 2]
```

In [ ]:
class LFUCache:
    """get, put in O(1). When full, evict the least frequently used key; ties -> least recently used."""

    def __init__(self, capacity):
        self.cap = capacity
        self.freq_of = {}                            # STATE: key -> use count
        self.bucket = defaultdict(OrderedDict)       # STATE: count -> {key: value}, oldest first
        self.min_freq = 0                            # STATE: the smallest non-empty count
        # INVARIANT: key lives only in bucket[freq_of[key]]; min_freq = smallest non-empty bucket

    def _touch(self, key):                           # helper: one more use of key; returns its value
        f = self.freq_of[key]
        value = self.bucket[f].pop(key)
        if not self.bucket[f]:
            del self.bucket[f]
            if self.min_freq == f:
                self.min_freq = f + 1                # the min floor emptied: key went to f + 1
        self.bucket[f + 1][key] = value              # newest on its new floor
        self.freq_of[key] = f + 1
        return value

    def _evict(self):
        victim, _ = self.bucket[self.min_freq].popitem(last=False)   # oldest of the rarest
        if not self.bucket[self.min_freq]:
            del self.bucket[self.min_freq]
        del self.freq_of[victim]

    def get(self, key):
        if key not in self.freq_of:
            return -1
        return self._touch(key)

    def put(self, key, value):
        if self.cap == 0:                            # edge: nothing can ever be stored
            return
        if key in self.freq_of:
            self._touch(key)                         # an update is a use too
            self.bucket[self.freq_of[key]][key] = value
            return
        if len(self.freq_of) == self.cap:
            self._evict()                            # evict FIRST: the newcomer must not be the victim
        self.bucket[1][key] = value
        self.freq_of[key] = 1
        self.min_freq = 1                            # a new key always has the smallest count


lfu = LFUCache(2)
lfu.put(1, 1)
lfu.put(2, 2)
print(lfu.get(1))                                # 1    (now key 1 has 2 uses, key 2 has 1)
lfu.put(3, 3)                                    # evicts 2, the key with the fewest uses
print(lfu.get(2), lfu.get(3))                    # -1 3
lfu.put(4, 4)                                    # 1 and 3 both have 2 uses; 1 is older -> evicted
print(lfu.get(1), lfu.get(3), lfu.get(4))        # -1 3 4

**Try it**
- Insert before evicting: move the eviction below the three insert lines (as `if len(self.freq_of) > self.cap: self._evict()`) and rerun. The last line prints `1 3 -1` instead of `-1 3 4`: the newcomer 4 sat alone on floor 1, so it was evicted instead of key 1.
- Delete `self.min_freq = 1` at the end of `put` and rerun: `KeyError: 'dictionary is empty'` at the first eviction. `min_freq` was still 0, a floor with no keys.
- Add `print({f: list(b) for f, b in lfu.bucket.items()}, lfu.min_freq)` after every call and watch keys climb one floor per use.
- `lfu.put(3, 30)` on an existing key counts as a use: key 3 climbs one floor and nothing is evicted.

### The tracker family

Google-style "implement a tracker" questions are small design problems over a **stream of events**: hits, log lines, values, check-ins, writes. Each comes down to two questions: *what must I remember to answer the queries?* and *what can I forget, and when?* Forgetting at the right moment is what keeps them O(1).

**Time windows.** Time only moves forward, so an event that left the window is gone for good: a deque drops it from the front, and a running total means no query ever re-adds anything.

In [ ]:
class HitCounter:                                # hits in the last 300 seconds
    def __init__(self):
        self.window = deque()                    # (timestamp, hits at that second), oldest first
        self.total = 0                           # INVARIANT: total = sum of the counts in window

    def _expire(self, now):
        while self.window and self.window[0][0] <= now - 300:
            self.total -= self.window.popleft()[1]   # too old, forever: time only moves forward

    def hit(self, timestamp):
        if self.window and self.window[-1][0] == timestamp:
            self.window[-1] = (timestamp, self.window[-1][1] + 1)   # same second: bump its count
        else:
            self.window.append((timestamp, 1))
        self.total += 1
        self._expire(timestamp)

    def getHits(self, timestamp):
        self._expire(timestamp)
        return self.total


class Logger:                                    # print each message at most once per 10 seconds
    def __init__(self):
        self.next_ok = {}                        # message -> earliest time it may print again

    def shouldPrintMessage(self, timestamp, message):
        if timestamp < self.next_ok.get(message, 0):
            return False
        self.next_ok[message] = timestamp + 10
        return True


class MovingAverage:                             # average of the last `size` values
    def __init__(self, size):
        self.size, self.window, self.total = size, deque(), 0

    def next(self, val):
        self.window.append(val)
        self.total += val
        if len(self.window) > self.size:
            self.total -= self.window.popleft()  # one value in, one value out
        return self.total / len(self.window)     # divide by the CURRENT length (warm-up)


hits = HitCounter()
for t in (1, 2, 3, 300):
    hits.hit(t)
print(hits.getHits(300), hits.getHits(301))      # 4 3
log = Logger()
print([log.shouldPrintMessage(t, "foo") for t in (1, 3, 11)])   # [True, False, True]
avg = MovingAverage(3)
print([round(avg.next(v), 2) for v in (1, 10, 3, 5)])           # [1.0, 5.5, 4.67, 6.0]

**Try it**
- Change `<=` to `<` in `_expire` and rerun: `getHits(301)` says 4. The hit at second 1 is exactly 300 seconds old, and the window is `(t - 300, t]`, so it must be out.
- Hit 1,000 times at the same second on a fresh counter: `len(window)` is 1. Merging equal timestamps caps the memory at 300 entries, however bursty the traffic.
- After the cell, `log.shouldPrintMessage(21, "foo")` is `True`: the last print was at 11, so 21 is exactly allowed again. Now change `<` to `<=` in the check and rerun: `[True, False, False]`. The call at 11, exactly 10 seconds after the first print, is wrongly refused.
- In `MovingAverage.next`, divide by `self.size` instead: the first two answers become 0.33 and 3.67 instead of 1.0 and 5.5. During warm-up the window is not full yet.

**Versions over time.** Writes arrive in time order, so each per-key history is sorted for free, and "the value as of time t" is a `bisect_right` followed by one step back.

In [ ]:
class TimeMap:                                   # the value of a key as of time t
    def __init__(self):
        self.times = defaultdict(list)           # key -> timestamps, increasing (writes come in time order)
        self.values = defaultdict(list)          # key -> values, aligned with times

    def set(self, key, value, timestamp):
        self.times[key].append(timestamp)        # appending keeps the list sorted: no sort needed
        self.values[key].append(value)

    def get(self, key, timestamp):
        i = bisect.bisect_right(self.times.get(key, []), timestamp)   # first time > timestamp
        return self.values[key][i - 1] if i else ""                     # one step back: last time <= timestamp


class SnapshotArray:                             # snapshots of an array, without copying it
    def __init__(self, length):
        self.history = [[(0, 0)] for _ in range(length)]   # per index: (snap_id, value), increasing
        self.snap_id = 0

    def set(self, index, val):
        h = self.history[index]
        if h[-1][0] == self.snap_id:
            h[-1] = (self.snap_id, val)          # same snapshot: overwrite, don't grow
        else:
            h.append((self.snap_id, val))

    def snap(self):
        self.snap_id += 1
        return self.snap_id - 1

    def get(self, index, snap_id):
        h = self.history[index]
        return h[bisect.bisect_right(h, (snap_id, math.inf)) - 1][1]   # last write at or before snap_id


tm = TimeMap()
tm.set("foo", "bar", 1)
tm.set("foo", "bar2", 4)
print(tm.get("foo", 1), tm.get("foo", 3), tm.get("foo", 4), repr(tm.get("foo", 0)))   # bar bar bar2 ''
arr = SnapshotArray(3)
arr.set(0, 5)
first = arr.snap()                               # snapshot 0: index 0 holds 5
arr.set(0, 6)
second = arr.snap()                              # snapshot 1: index 0 holds 6
print(first, second, arr.get(0, 0), arr.get(0, 1), arr.get(2, 1), arr.history[0])   # 0 1 5 6 0 [(0, 5), (1, 6)]

**Try it**
- Use `bisect_left` in `TimeMap.get` and rerun: exact timestamps are skipped, so `get("foo", 1)` comes back empty and `get("foo", 4)` returns `bar`.
- In `SnapshotArray.get`, probe with `(snap_id, 0)` instead of `(snap_id, math.inf)` and rerun: `get(0, 1)` returns 5. The write made during snapshot 1 sorts *after* the probe, so it is missed.
- Call `arr.set(1, 7)` three times in a row: `arr.history[1]` is `[(0, 0), (2, 7)]`, one pair per snapshot, not per write.
- `tm.get("nope", 5)` is `''`: an unknown key is just an empty history.

**Two lifetimes.** Some data lives briefly (an open journey, the forward history of a browser tab), some lives forever but compresses (a route's total time and trip count). Give each lifetime its own structure.

In [ ]:
class UndergroundSystem:
    def __init__(self):
        self.open_trips = {}                     # id -> (start station, check-in time): short-lived
        self.stats = {}                          # (start, end) -> [total time, trips]: kept forever

    def checkIn(self, id, stationName, t):
        self.open_trips[id] = (stationName, t)

    def checkOut(self, id, stationName, t):
        start, t0 = self.open_trips.pop(id)      # the journey is over: forget it
        cell = self.stats.setdefault((start, stationName), [0, 0])
        cell[0] += t - t0
        cell[1] += 1

    def getAverageTime(self, startStation, endStation):
        total, trips = self.stats[(startStation, endStation)]
        return total / trips


class BrowserHistory:
    def __init__(self, homepage):
        self.pages, self.cur, self.last = [homepage], 0, 0   # last = the newest page still reachable

    def visit(self, url):
        self.cur += 1
        if self.cur == len(self.pages):
            self.pages.append(url)
        else:
            self.pages[self.cur] = url           # overwrite stale forward history in place
        self.last = self.cur                     # forward history is gone: a bound, not a deletion

    def back(self, steps):
        self.cur = max(0, self.cur - steps)
        return self.pages[self.cur]

    def forward(self, steps):
        self.cur = min(self.last, self.cur + steps)
        return self.pages[self.cur]


u = UndergroundSystem()
u.checkIn(45, "Leyton", 3)
u.checkIn(27, "Leyton", 10)
u.checkOut(45, "Waterloo", 15)
u.checkOut(27, "Waterloo", 20)
print(u.getAverageTime("Leyton", "Waterloo"), u.open_trips)   # 11.0 {}
b = BrowserHistory("home")
for url in ("a", "b", "c"):
    b.visit(url)
print(b.back(2), b.forward(1), b.back(1))        # a b a
b.visit("x")                                     # overwrites "b"; "c" stays in the list, unreachable
print(b.forward(5), b.pages, b.last)             # x ['home', 'a', 'x', 'c'] 2

**Try it**
- In `forward`, clamp with `len(self.pages) - 1` instead of `self.last` and rerun: `forward(5)` lands on `c`, a page that `visit` was supposed to clear.
- In `checkOut`, read with `self.open_trips[id]` instead of `pop`: the average is unchanged, but `u.open_trips` keeps both finished journeys forever.
- `u.getAverageTime("Waterloo", "Leyton")` raises `KeyError`: a route is an ordered pair, so the dict key is the tuple `(start, end)`.

**O(1) tricks.** RandomizedSet keeps a dense list (so a random index is a uniform pick) plus value → index (so membership is O(1)), and deletes by moving the last value into the hole. FreqStack keeps one stack per frequency: a value pushed three times sits on floors 1, 2 and 3, and popping it from the top floor is exactly "its frequency drops by one".

In [ ]:
class RandomizedSet:
    def __init__(self):
        self.vals = []                           # dense list: random.choice needs no holes
        self.pos = {}                            # value -> its index in vals
        # INVARIANT: vals[pos[v]] == v for every stored v

    def insert(self, val):
        if val in self.pos:
            return False
        self.pos[val] = len(self.vals)
        self.vals.append(val)
        return True

    def remove(self, val):
        if val not in self.pos:
            return False
        i, last = self.pos[val], self.vals[-1]
        self.vals[i], self.pos[last] = last, i   # move the last value into the hole
        self.vals.pop()
        del self.pos[val]                        # AFTER the move: works even when val is the last one
        return True

    def getRandom(self):
        return random.choice(self.vals)


class FreqStack:                                 # pop the most frequent; ties -> the most recent
    def __init__(self):
        self.freq = defaultdict(int)             # value -> current frequency
        self.group = defaultdict(list)           # f -> stack of values in the order they REACHED f
        self.max_freq = 0

    def push(self, val):
        self.freq[val] += 1
        f = self.freq[val]
        self.group[f].append(val)
        self.max_freq = max(self.max_freq, f)

    def pop(self):
        val = self.group[self.max_freq].pop()
        if not self.group[self.max_freq]:
            self.max_freq -= 1                   # frequencies are reached one step at a time
        self.freq[val] -= 1
        return val


random.seed(0)
rs = RandomizedSet()
print(rs.insert(5), rs.insert(8), rs.insert(2), rs.remove(5), rs.vals, rs.pos)   # True True True True [2, 8] {8: 1, 2: 0}
print(sorted({rs.getRandom() for _ in range(50)}))                              # [2, 8]
fs = FreqStack()
for v in (5, 7, 5, 7, 4, 5):
    fs.push(v)
print([fs.pop() for _ in range(4)], dict(fs.group))   # [5, 7, 5, 4] {1: [5, 7], 2: [], 3: []}

**Try it**
- Move `del self.pos[val]` up, before the line that fills the hole, then remove the only element: `r = RandomizedSet(); r.insert(1); r.remove(1)`. Now `r.pos` is `{1: 0}`, a ghost entry, and `r.insert(1)` returns `False`.
- Print `dict(fs.group)` right after the six pushes: `{1: [5, 7, 4], 2: [5, 7], 3: [5]}`. A value pushed three times lives on three floors.
- After the cell, `fs.push(7)` then `fs.pop()` returns 7: it still had frequency 1, so the new push lifted it to floor 2, the top.
- `Counter(rs.getRandom() for _ in range(3000))`: each of the two values comes up about 1,500 times.

### Where it goes wrong

1. **The node does not store its key.** Eviction finds `tail.prev` but cannot delete it from the dict.
2. **Updating one structure and forgetting the other.** Every helper must leave the invariant true: unlink *and* `del map[key]`, pop the deque *and* subtract from the total.
3. **`put` of an existing key that only updates the value.** It must refresh recency (LRU) or count as a use (LFU) as well.
4. **Insert-then-evict in LFU.** The newcomer has the smallest count and can evict itself. LFU must evict first; LRU may do either (inserting first also makes capacity 0 safe).
5. **Forgetting `min_freq = 1`** after inserting a new key in LFU.
6. **A singly linked list.** Unlinking needs the previous node; only a doubly linked list gives it in O(1). Sentinels remove every `None` check.
7. **Window boundaries.** "The last 300 seconds" is `(t - 300, t]`: pop while `timestamp <= now - 300`. A logger's "10 seconds" allows the same message again at exactly `t + 10`.
8. **`bisect_left` for "as of time t".** Use `bisect_right` (or probe with `(t, inf)`), or an exact match is skipped.
9. **Swap-with-last deletion in the wrong order.** Write `pos[last] = i` before `del pos[val]`, or removing the last element leaves a ghost.
10. **Trusting a lazily deleted heap top.** Validate the top before using it (Dinner Plate Stacks, Movie Rental).

### Edge cases to say out loud

Capacity 0 and 1 · `get` of a missing key · `put` of an existing key (refresh, no eviction) · an empty structure · a burst of events at one timestamp · a query before the first timestamp · removing the only (= last) element · back/forward with nowhere to go.

In [ ]:
c = LRUCache(1)
c.put(5, 5)
c.put(5, 6)                                      # update an existing key: no eviction
assert c.get(5) == 6
c.put(7, 7)                                      # evicts 5
assert c.get(5) == -1 and c.get(7) == 7
z = LRUCache(0)                                  # capacity 0: insert, then evict at once
z.put(1, 1)
assert z.get(1) == -1 and z.map == {}
lf = LFUCache(1)
lf.put(1, 1)
lf.get(1)                                        # key 1 now has 2 uses
lf.put(2, 2)                                     # evicts 1, not the newcomer
assert lf.get(1) == -1 and lf.get(2) == 2
assert LFUCache(0).get(3) == -1
assert TimeMap().get("missing", 9) == ""
burst = HitCounter()
for _ in range(5):
    burst.hit(10)
assert burst.getHits(309) == 5 and burst.getHits(310) == 0 and len(burst.window) == 0
r = RandomizedSet()
assert r.insert(7) and r.remove(7) and r.vals == [] and r.pos == {}   # remove the only (= last) element
assert r.insert(7) and r.getRandom() == 7
home = BrowserHistory("home")
assert home.back(3) == "home" and home.forward(3) == "home"           # nowhere to go
print("edge cases pass")

**Try it**
- Write your own case: `c2 = LRUCache(2)`, `put(1, 1)`, `put(2, 2)`, `put(1, 10)`, `put(3, 3)`, then `assert c2.get(2) == -1 and c2.get(1) == 10`.
- What should `HitCounter().getHits(5)` return with no hits at all? Predict, then assert it (0).
- A window of one is always the latest value: `m1 = MovingAverage(1)`, then `[m1.next(v) for v in (4, -2)]` is `[4.0, -2.0]`.

### Variations

| Variation | What changes from the template | Problems |
|---|---|---|
| **Count buckets in a linked list** | LRU's list, but each node is a *count* holding a set of keys; ±1 moves a key to the neighbouring bucket | All O'one Data Structure (432) |
| **One stack per frequency** | the top of the highest stack is "most frequent, most recent" | Maximum Frequency Stack (895) |
| **A lazy flag instead of a mass update** | keep a "flipped" flag and a count; read every bit through the flag | Design Bitset (2166) |
| **Heap of candidates, validated lazily** | a min-heap of indices that may have room; check the top before trusting it | Dinner Plate Stacks (1172) |
| **Heaps + version stamps** | every state change pushes a fresh entry; stale entries are skipped when they surface | Design Movie Rental System (1912) |
| **Sorted boundaries + bisect** | intervals stay disjoint and sorted; a change only touches its neighbours | Range Module (715), Data Stream as Disjoint Intervals (352) |
| **Hashing by hand** | an array of short chains at `hash(key) % B`; double B and rehash when chains get long | Design HashMap (706) |
| **Trie of dicts** | each path component is an edge; a node is a directory (children) or a file (content) | Design In-Memory File System (588) |
| **Express lanes** | a sorted linked list plus random-height towers; a search drops a level when it cannot move right | Design Skiplist (1206) |
| **Positions + random sampling** | value → sorted positions; sample a few indices, verify each with two bisects | Online Majority Element In Subarray (1157) |
| **Fenwick tree** | point update and prefix sum in O(log n) per axis; a rectangle is 4 prefixes | Range Sum Query 2D - Mutable (308) |
| **A snapshot next to each entry** | push (value, min so far) together | Min Stack (155), in Stacks & Queues |
| **Two heaps around the middle** | a max-heap for the low half, a min-heap for the high half, sizes within one | Find Median from Data Stream (295), in Heaps |

**A lazy flag** (Design Bitset): flipping every bit is a change of *viewpoint*, not of data. Store one flag and read each bit through it; keep the count of ones up to date so `all`, `one` and `count` are O(1).

In [ ]:
class Bitset:
    def __init__(self, size):
        self.bits = [0] * size                   # physical bits; the bit you SEE = physical ^ flipped
        self.flipped = 0                         # one flag stands for "every bit is inverted"
        self.ones = 0                            # INVARIANT: the number of 1s you see

    def fix(self, idx):                          # set to 1
        if self.bits[idx] ^ self.flipped == 0:
            self.bits[idx] ^= 1
            self.ones += 1

    def unfix(self, idx):                        # set to 0
        if self.bits[idx] ^ self.flipped == 1:
            self.bits[idx] ^= 1
            self.ones -= 1

    def flip(self):                              # O(1): change the lens, not the bits
        self.flipped ^= 1
        self.ones = len(self.bits) - self.ones

    def all(self):
        return self.ones == len(self.bits)

    def one(self):
        return self.ones > 0

    def count(self):
        return self.ones

    def toString(self):
        return "".join(str(b ^ self.flipped) for b in self.bits)


bs = Bitset(5)
bs.fix(3)
bs.fix(1)
bs.flip()
print(bs.toString(), bs.count(), bs.all())       # 10101 3 False
bs.unfix(0)
bs.flip()
print(bs.toString(), bs.bits, bs.flipped)        # 11010 [1, 1, 0, 1, 0] 0

**Try it**
- Make `fix` toggle and count unconditionally (delete its `if` and dedent the two lines under it): `b2 = Bitset(2); b2.fix(0); b2.fix(0)` leaves `toString()` at `"00"` while `count()` says 2. Fixing an already-set bit must do nothing.
- Use `str(b)` in `toString` (forget the flag) and rerun: the first line shows `01010` instead of `10101`.
- Call `bs.flip()` twice and compare `bs.bits` before and after: identical. Only the flag and the count changed.

**Sorted boundaries** (Range Module): a union of disjoint half-open blocks `[l, r)` is fully described by its sorted boundaries `[l0, r0, l1, r1, ...]`. The parity of a bisect position tells you where you are:

```text
ends = [10, 14, 16, 20]        tracked: [10, 14) and [16, 20)

  x:                    5     12     15     18     25
  bisect_right(ends,x): 0     1      2      3      4
  inside a block?       no    YES    no     YES    no        odd position = inside
```

In [ ]:
class RangeModule:
    def __init__(self):
        self.ends = []                           # sorted boundaries [l0, r0, l1, r1, ...] of disjoint [l, r)

    def addRange(self, left, right):
        i = bisect.bisect_left(self.ends, left)  # even i: left is in a gap; odd i: inside a block
        j = bisect.bisect_right(self.ends, right)
        new = ([left] if i % 2 == 0 else []) + ([right] if j % 2 == 0 else [])
        self.ends[i:j] = new                     # every boundary inside [left, right] is swallowed

    def removeRange(self, left, right):
        i = bisect.bisect_left(self.ends, left)
        j = bisect.bisect_right(self.ends, right)
        new = ([left] if i % 2 == 1 else []) + ([right] if j % 2 == 1 else [])
        self.ends[i:j] = new                     # cut the slice out; close/reopen where a block was cut

    def queryRange(self, left, right):
        i = bisect.bisect_right(self.ends, left)
        j = bisect.bisect_left(self.ends, right)
        return i == j and i % 2 == 1             # both ends inside the same block


rm = RangeModule()
rm.addRange(10, 20)
rm.removeRange(14, 16)
print(rm.ends, rm.queryRange(10, 14), rm.queryRange(13, 15), rm.queryRange(16, 17))   # [10, 14, 16, 20] True False True
rm.addRange(14, 16)                              # fill the hole: the two blocks merge back
print(rm.ends)                                   # [10, 20]

**Try it**
- Touching blocks merge: after the cell, `rm.addRange(20, 25)` makes `rm.ends` `[10, 25]`, because `bisect_left` puts 20 inside the block that ends there.
- Use `bisect_right` for `left` in `addRange` and rerun: the last line shows `[10, 14, 14, 20]`. Filling the hole left a zero-width seam at 14, so the blocks touch but never merge, and `rm.queryRange(13, 15)` wrongly says `False`.
- Half-open in action: with `ends == [10, 20]`, `rm.queryRange(10, 20)` is `True` but `rm.queryRange(10, 21)` is `False`.

### Say it in the interview

> "Let me list the operations and the cost each needs: `get` and `put`, both O(1). Finding a key in O(1) means a dict. Keeping recency order with O(1) move-to-front and O(1) remove-oldest means a doubly linked list, and the dict will point at its nodes. The invariant is that the dict and the list always hold the same keys. I'll write two helpers, unlink and push-front, so `get` and `put` are a few lines each. Every operation is O(1) time, and the space is O(capacity)."

While coding, keep the invariant comment visible in `__init__` and, after each public method, say which helper restored it. If the interviewer allows libraries, `OrderedDict` with `move_to_end` and `popitem(last=False)` is the short version; offer to write the linked list yourself to show you know what it does.

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| All O'one Data Structure | `design/all_oone_data_structure.py` | doubly linked list of count buckets; a ±1 change moves a key to the neighbouring bucket |
| Data Stream as Disjoint Intervals | `design/data_stream_as_disjoint_intervals.py` | sorted starts/ends + bisect_right; a new value can only extend or bridge its two neighbours |
| Design Bitset | `design/design_bitset.py` | flip is a lazy flag (seen bit = stored bit XOR flag) plus a maintained count of ones |
| Design Browser History | `design/design_browser_history.py` | list + cursor + logical end; visit overwrites, forward clamps to the end |
| Design HashMap | `design/design_hashmap.py` | buckets of chains at hash % B; double B and rehash when the load factor passes 2 |
| Design Hit Counter | `design/design_hit_counter.py` | deque of (timestamp, count) + running total; pop the front once it is 300 s old |
| Design In-Memory File System | `design/design_in_memory_file_system.py` | trie keyed by path components; a node is a directory (children) or a file (content) |
| Design Movie Rental System | `design/design_movie_rental_system.py` | a heap per movie and one for rentals; lazy deletion with version stamps |
| Design Skiplist | `design/design_skiplist.py` | sorted list with random-height express lanes; record where the search turns down on each level |
| Design Underground System | `design/design_underground_system.py` | two dicts: open journeys by id, and (start, end) → [total time, trips] |
| Dinner Plate Stacks | `design/dinner_plate_stacks.py` | min-heap of indices that may have room, validated lazily; trim empty stacks on the right |
| Insert Delete GetRandom O(1) | `design/insert_delete_getrandom_o1.py` | list + value→index; delete by moving the last value into the hole |
| LFU Cache | `design/lfu_cache.py` | count → OrderedDict buckets + min_freq, which only resets to 1 or steps up by 1 |
| Logger Rate Limiter | `design/logger_rate_limiter.py` | dict message → next allowed timestamp |
| LRU Cache | `design/lru_cache.py` · `practice/simple/24_lru_cache.py` | dict key → node + doubly linked list with sentinels; the node stores its key for eviction |
| Maximum Frequency Stack | `design/maximum_frequency_stack.py` | one stack per frequency; pop from the stack of the max frequency |
| Moving Average from Data Stream | `design/moving_average_from_data_stream.py` | deque of the last `size` values + running sum |
| Online Majority Element In Subarray | `design/online_majority_element_in_subarray.py` | value → sorted positions; random samples verified with two bisects |
| Range Module | `design/range_module.py` | one flat sorted boundary list; an odd bisect position means "inside a block" |
| Range Sum Query 2D - Mutable | `design/range_sum_query_2d_mutable.py` | 2D Fenwick tree; update by the delta, query as 4 prefix rectangles |
| Snapshot Array | `design/snapshot_array.py` | per-index list of (snap_id, value); bisect_right for the last write ≤ snap_id |
| Time Based Key-Value Store | `design/time_based_key_value_store.py` | per-key timestamps appended in order; bisect_right − 1 |

### Self-check

1. The dict already maps key → node. Why must each LRU node also store its key?
<details><summary>Answer</summary>Eviction starts from the list side: it finds the oldest node as <code>tail.prev</code> and must then delete that node's entry from the dict. Without the key inside the node there is no O(1) way to know which dict entry to delete.</details>

2. LRU may insert the new node first and evict afterwards; LFU must evict first. Why the difference?
<details><summary>Answer</summary>In LRU the newcomer is the most recently used item, so it can never be the eviction victim (unless the capacity is 0, where evicting it is correct). In LFU the newcomer has a count of 1, the smallest possible, so if it is inserted first it can be the oldest item on the lowest floor and evict itself.</details>

3. In `RandomizedSet.remove`, what goes wrong if `del pos[val]` runs before `pos[last] = i`?
<details><summary>Answer</summary>When <code>val</code> is the last element, <code>last == val</code>, so <code>pos[last] = i</code> re-creates the entry that was just deleted. The set then believes <code>val</code> is still present (a ghost), and the next <code>insert(val)</code> wrongly returns False.</details>

4. HitCounter throws expired hits away for good. Why is that safe, and what does it buy?
<details><summary>Answer</summary>Timestamps never decrease, so a hit that is 300 seconds old now will be even older at every later query: it can never count again. Each hit is appended once and popped once, so all the eviction work is O(1) amortized per hit, and the running total answers <code>getHits</code> without scanning.</details>